# Test matrizActividades.py


In [ ]:

"""Notebook para probar las funcionalidades y pulir el desempeño de la función que genera la matriz de actividades."""

## 1. Configuración global

In [1]:

import os
import sys
import time
from pathlib import Path
import pandas as pd
import re
import json
import pickle
import logging
import pymongo
from pymongo.errors import ConnectionFailure
from deltalake import DeltaTable, write_deltalake
from pprint import pprint
from datetime import datetime

from eerssa.secret import Keys

logging.basicConfig(level=logging.INFO)


# --- MongoDB Connection ---
# It's better to establish the connection once and keep it open for the app's lifetime.
# We will also exit if the connection fails, as the consumer can't do its job without it.
uri = Keys.MONGO_KEY.value
client = None  # Initialize client to None
db_eerssa = None
CurrentCollection = None
ReloadCollection = None


try:
    # Add a timeout to avoid blocking indefinitely
    client = pymongo.MongoClient(uri, serverSelectionTimeoutMS=5000)
    # The ping command is cheap and does not require auth.
    client.admin.command('ping')
    db_eerssa = client.eerssa                   # Base de datos EERSSA
    CurrentCollection = db_eerssa.ot_v22        # Coleccion actual
    ReloadCollection  = db_eerssa.ot_reemplazo  # Aqui se cargan OTs repetidas
    logging.info(":::: Conexion exitosa con MongoDB ::::")
    
except ConnectionFailure as e:
    logging.error(f"\n\n ><><> Error de conexion a MongoDB: {e}")
    sys.exit(1) # Exit the script if we can't connect to MongoDB, as it's a critical dependency.


# Ubicación del directorio DELTA LAKE TABLE
table_path = "./test/deltalake_2025"

# DELTA LAKE Connection

# Verify the existence of the DELTA LAKE table
if not DeltaTable.is_deltatable(table_path):
    print(
        f"No se ha encontrado la base de datos PARQUET-DELTALAKE en la direccion:\n NO_DELTA_LAKE : {table_path}" )
else:
    print(f"Conectado a la tabla Delta Lake en: {table_path}")


Success!!!


INFO:root::::: Conexion exitosa con MongoDB ::::


Conectado a la tabla Delta Lake en: ./test/deltalake_2025


In [10]:
## RECARGAR LAS LIBRERIAS DINAMICAMENTE
from importlib import reload
from eerssa import gestionOT as OrdenTrabajo                   # Convert from PDF_ot to obj_ot
from eerssa import matrizActividades as Actividades     # process ot.data["actividades"]

In [24]:
reload( OrdenTrabajo )
reload( Actividades  )

<module 'eerssa.matrizActividades' from '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/eerssa/matrizActividades.py'>

## Obtener Ordenes de trabajo desde MongoDB

In [3]:
test_json = CurrentCollection.find_one({"id_ot":157727}) 

In [4]:
type(test_json)

dict

In [ ]:
test_obj = OrdenTrabajo.GestionOt.from_dict(test_json)

activ = test_obj.data['actividades']
df_test = pd.DataFrame( activ)
df_test


,Item,Actividad,Evento,Ali,Alimentador,Tipo,InicioEvento,FinEvento
0,1,PROG,En la agencia de la EERSSA Guayzimi se coordin...,·,·,RUTINARIA,2025-07-23 08:00:00,2025-07-23 09:00:00
1,2,PROG,"RECLAMO (compañero Asdrúbal Cachay) Atendido, ...",ALI,Paquisha,CORRECTIVO,2025-07-23 09:00:00,2025-07-23 09:45:00
2,3,TRANSP,Traslado desde Guayzimi hacia el barrio San Ma...,·,·,TRANSPORTE,2025-07-23 09:45:00,2025-07-23 10:14:00
3,4,PROG,"INC. #. 1103773287. Atendido, se cambia lámpar...",ALI,Paquisha,CORRECTIVO,2025-07-23 10:14:00,2025-07-23 11:10:00
4,5,PROG,"INC. #. 1103773287. Atendido, se cambia lámpar...",ALI,Paquisha,CORRECTIVO,2025-07-23 11:10:00,2025-07-23 11:30:00
5,6,PROG,"INC. #. 1103773287. Atendido, se cambia lámpar...",ALI,Paquisha,CORRECTIVO,2025-07-23 11:30:00,2025-07-23 12:15:00
6,7,TRANSP,Traslado desde el barrio San Manuel de la parr...,·,·,TRANSPORTE,2025-07-23 12:15:00,2025-07-23 12:45:00
7,8,ALIMEN,Lunch en Guayzimi,·,·,ALIMENTACI,2025-07-23 12:45:00,2025-07-23 13:45:00
8,9,TRANSP,Traslado desde Guayzimi hacia Zurmi,·,·,TRANSPORTE,2025-07-23 13:45:00,2025-07-23 14:00:00
9,10,PROG,"OFICIO. Atendido, se cambia lámpara, de 100 W ...",ALI,Paquisha,CORRECTIVO,2025-07-23 14:00:00,2025-07-23 14:15:00


In [25]:
test_matriz = Actividades.ConvertirOT_a_ActividadesCSV(test_obj)

  >> Regreso de Organizar Actividades, tipo de objeto: <class 'pandas.core.frame.DataFrame'>


/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/.venv/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MultinomialNB from version 1.4.1.post1 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/.venv/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator CountVectorizer from version 1.4.1.post1 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
